# 07 - Stress testing and perturbation analysis (Test 7)

**Research question.** Which models degrade gracefully when volatility, correlation or the data themselves are disturbed in controlled ways?

Perturbations (all applied to the *return sample*, so each model re-estimates with its own estimator):
- mean noise of size $k\times$SE
- volatility scaling of selected assets
- common-factor correlation shock
- injected extreme observations
- historical scenario replay of fixed weights

In [1]:
import sys, warnings
import os
from pathlib import Path
try:  
    os.chdir(Path(__vsc_ipynb_file__).resolve().parent)
except NameError:  
    pass
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from src import data_processing as dp, covariance_estimators as ce, portfolio_models as pm
from src import stability_tests as st, regime_detection as rd, stress_testing as sx, evaluation as ev
from src.backtesting import walk_forward, make_models, in_sample_vs_out_of_sample
plt.rcParams["figure.dpi"] = 110
_candidates = ["returns_simple_daily.csv"]
_path = next((Path("../data/processed") / f for f in _candidates if (Path("../data/processed") / f).exists()), None)
if _path is None:
    raise FileNotFoundError("Run notebook 01 first: expected ../data/processed/returns_simple_daily.csv")
returns = pd.read_csv(_path, index_col=0, parse_dates=True)
print("loaded:", _path.name)
X = returns.to_numpy()
WINDOW = 504
TABLE_DIR = Path("../results/tables")
TABLE_DIR.mkdir(parents=True, exist_ok=True)
print(returns.shape, returns.index[0].date(), "->", returns.index[-1].date())

loaded: returns_simple_daily.csv
(3519, 5) 2012-01-04 -> 2025-12-31


In [2]:
last = X[-WINDOW:]
core = {m: pm.MODELS[m] for m in pm.CORE_TEN}

## 1. Correlation shock

In [3]:
rows = {}
for shift in [0.0, 0.25, 0.5, 1.0]:
    Xs = sx.shift_correlation(last, shift)
    C = ce.to_correlation(ce.sample_covariance(Xs)); n = len(C)
    rows[shift] = {"mean_corr_after": (C.sum() - n) / (n * (n - 1)),
                   "cond_number": ce.condition_number(ce.sample_covariance(Xs))}
correlation_shift_summary = pd.DataFrame(rows).T.round(3)
correlation_shift_summary.to_csv(TABLE_DIR / "correlation_shift_covariance_diagnostics.csv", index_label="correlation_shift")
print(correlation_shift_summary)
res = sx.compare_models_under_perturbation(core, last, lambda x, lev, rng: sx.shift_correlation(x, lev), level=0.5, n_rep=1)
correlation_shift_results = res.round(3)
correlation_shift_results.to_csv(TABLE_DIR / "correlation_shift_model_results.csv", index_label="model")
correlation_shift_results

      mean_corr_after  cond_number
0.00            0.357       33.810
0.25            0.538       58.518
0.50            0.664       90.630
1.00            0.808      176.545


,mean_turnover_equiv,p95_turnover_equiv,mean_vol_change,n_ok
equal_weight,0.000,0.000,0.000,1.0
gmv_long_only,0.078,0.078,0.021,1.0
mean_variance,0.227,0.227,0.053,1.0
max_sharpe,0.500,0.500,0.697,1.0
min_vol_capped,0.114,0.114,0.007,1.0
ledoit_wolf_minvar,0.167,0.167,-0.004,1.0
robust_mv,0.127,0.127,-0.027,1.0
risk_parity,0.015,0.015,-0.002,1.0
black_litterman,0.055,0.055,-0.049,1.0
cond_vol_minvar,0.028,0.028,0.005,1.0


**Interpret:** why does risk parity or GMV react differently from max-Sharpe? Is the "vol_change" column risk **reduction** or **increase**, and relative to which covariance?

## 2. Volatility shock on one asset class

In [4]:
idx = [0]  # first asset; change to target e.g. an equity or a commodity ETF
res = sx.compare_models_under_perturbation(core, last, lambda x, lev, rng: sx.scale_volatility(x, lev, idx), level=2.0, n_rep=1)
volatility_scale_results = res.round(3)
volatility_scale_results.to_csv(TABLE_DIR / "volatility_scale_model_results.csv", index_label="model")
volatility_scale_results

,mean_turnover_equiv,p95_turnover_equiv,mean_vol_change,n_ok
equal_weight,0.000,0.000,0.000,1.0
gmv_long_only,0.054,0.054,0.015,1.0
mean_variance,0.210,0.210,0.055,1.0
max_sharpe,0.235,0.235,0.064,1.0
min_vol_capped,0.224,0.224,0.035,1.0
ledoit_wolf_minvar,0.132,0.132,0.064,1.0
robust_mv,0.135,0.135,0.123,1.0
risk_parity,0.061,0.061,-0.031,1.0
black_litterman,0.118,0.118,-0.100,1.0
cond_vol_minvar,0.000,0.000,0.000,1.0


## 3. Outlier contamination

In [5]:
for common in (True, False):
    res = sx.compare_models_under_perturbation(core, last,
        lambda x, lev, rng, c=common: sx.inject_outliers(x, 3, lev, rng, common=c), level=8.0, n_rep=40)
    shock_type = "common" if common else "idiosyncratic"
    print("COMMON shock" if common else "IDIOSYNCRATIC shock")
    outlier_results = res.round(3)
    outlier_results.to_csv(TABLE_DIR / f"{shock_type}_outlier_model_results.csv", index_label="model")
    display(outlier_results)

COMMON shock


,mean_turnover_equiv,p95_turnover_equiv,mean_vol_change,n_ok
equal_weight,0.000,0.000,0.000,40.0
gmv_long_only,0.078,0.078,0.021,40.0
mean_variance,0.089,0.180,-0.031,40.0
max_sharpe,0.275,0.507,0.382,40.0
min_vol_capped,0.105,0.114,0.007,40.0
ledoit_wolf_minvar,0.072,0.082,-0.020,40.0
robust_mv,0.056,0.247,-0.014,40.0
risk_parity,0.021,0.029,0.020,40.0
black_litterman,0.014,0.061,-0.012,40.0
cond_vol_minvar,0.009,0.028,0.001,40.0


IDIOSYNCRATIC shock


,mean_turnover_equiv,p95_turnover_equiv,mean_vol_change,n_ok
equal_weight,0.000,0.000,0.000,40.0
gmv_long_only,0.014,0.033,0.001,40.0
mean_variance,0.045,0.114,0.010,40.0
max_sharpe,0.129,0.265,0.031,40.0
min_vol_capped,0.042,0.078,0.002,40.0
ledoit_wolf_minvar,0.022,0.034,0.012,40.0
robust_mv,0.084,0.193,0.012,40.0
risk_parity,0.012,0.021,0.003,40.0
black_litterman,0.037,0.219,-0.028,40.0
cond_vol_minvar,0.039,0.297,0.033,40.0


**Interpret:** three 8-sigma days in 504 observations is a small contamination. Which models are most sensitive, and what does that say about the reliance of each on *second moments* estimated by plain averaging?

## 4. Perturbing the covariance spectrum directly

In [6]:
rng = np.random.default_rng(0)
S0 = ce.sample_covariance(last)
w0 = pm.gmv_long_only(last)
rows = []
for noise in [0.05, 0.1, 0.25, 0.5]:
    out = []
    for _ in range(50):
        Sp = ce.nearest_psd(sx.perturb_covariance_eigen(last, noise, rng))
        w = pm.gmv_long_only(last, cov_estimator=lambda r, Sp=Sp: Sp)
        out.append((0.5 * np.abs(w - w0).sum(), (w @ S0 @ w) / (w0 @ S0 @ w0)))
    o = np.array(out); rows.append({"eig_noise": noise, "turnover_equiv": o[:, 0].mean(), "var_inflation_under_S0": o[:, 1].mean()})
eigenvalue_perturbation_summary = pd.DataFrame(rows).set_index("eig_noise").round(4)
eigenvalue_perturbation_summary.to_csv(TABLE_DIR / "covariance_eigenvalue_perturbation_summary.csv", index_label="eig_noise")
eigenvalue_perturbation_summary

,turnover_equiv,var_inflation_under_S0
eig_noise,,
0.05,0.0079,1.0007
0.10,0.0161,1.0027
0.25,0.0384,1.0137
0.50,0.0545,1.0309


**Interpret:** this is *mathematical* sensitivity (a controlled perturbation of the exact input), not sampling noise. How does variance inflation under the original covariance grow with perturbation size? Is the optimiser's loss flat near the optimum (second-order) while weights move a lot? Explain with the quadratic structure of the objective.

## 5. Historical scenarios (fixed weights from the preceding window)

In [7]:
out = {}
for name, (s, e) in rd.STRESS_WINDOWS.items():
    sub = returns.loc[s:e]
    if len(sub) < 20: continue
    est = returns.loc[:s].iloc[:-1].tail(WINDOW)
    if len(est) < WINDOW: continue
    row = {}
    for m in ["equal_weight", "gmv_long_only", "max_sharpe", "ledoit_wolf_minvar", "risk_parity", "robust_mv"]:
        w = pd.Series(pm.MODELS[m](est.values), index=returns.columns)
        row[m] = sx.historical_scenario_returns(w, returns, s, e)["cum_return"]
    out[name] = row
if out:
    stress_window_returns = pd.DataFrame(out).T.round(3)
    stress_window_returns.to_csv(TABLE_DIR / "historical_stress_window_returns.csv", index_label="stress_window")
    stress_window_returns
else:
    print("No stress window with enough prior history in this sample.")

**Caveat to discuss:** weights are fixed from information *before* each episode (no look-ahead), but only one outcome per episode is observed - five stress windows are far too few for statistical conclusions. State what you can and cannot conclude.

# Interpretation

Based on the saved outputs in `07_stress_testing.ipynb`, this analysis asks how portfolio weights respond to controlled changes in the input data—not which portfolio performed best in a real crisis. It uses five ETFs and a 504-observation window from the end of the return sample (2012–2025).

### How to read the tables

- **Turnover equivalent** is half the L1 distance between the original and perturbed weights. For example, `0.10` means a 10% one-way allocation change—not transaction costs or measured realized turnover.
- **Mean vol change** compares the perturbed portfolio’s volatility with its original weights’ volatility, both evaluated using the *unperturbed sample covariance*. Positive means higher risk under that reference covariance; negative means lower risk there. It does **not** measure risk under the shocked data.
- The correlation and single-asset volatility experiments use **one run** each, so their reported means and 95th percentiles are the same observation. Outlier results average 40 runs.

### 1. Correlation shock

The shock raises average pairwise correlation from **0.357** to **0.664** at level 0.5, while the covariance condition number rises from **33.81** to **90.63**. At level 1.0, average correlation reaches **0.808** and the condition number **176.55**. In short, the stronger common factor makes the covariance estimate increasingly ill-conditioned.

At level 0.5, **max-Sharpe is the clear outlier**: its weights move by 50% turnover equivalent, and their volatility under the original covariance is 69.7% higher than for its original portfolio. Mean-variance also moves substantially (22.7%) and has a 5.3% higher reference volatility. By contrast, risk parity moves only 1.5%, conditional-volatility minimum variance 2.8%, and Black–Litterman 5.5%; their reference volatility is slightly lower.

This suggests that, in this window and setup, max-Sharpe is especially exposed to changing inputs, while risk parity and the other more stable methods change their allocations less. The result is consistent with max-Sharpe depending on estimated returns as well as covariance, but it is a single controlled comparison—not evidence that it will always behave this way.

### 2. Volatility shock to SPY

The notebook scales deviations from the mean for **SPY** by a factor of two. Mean-variance, max-Sharpe, and capped minimum-volatility make the largest allocation changes (21.0%, 23.5%, and 22.4% turnover equivalent). Risk parity and GMV move less (6.1% and 5.4%). Conditional-volatility minimum variance shows no weight change in this run.

Most changed portfolios have higher reference volatility; risk parity and Black–Litterman are exceptions, at −3.1% and −10.0%. Those negative values mean their new weights happen to be lower-risk under the *original* covariance—not that doubling SPY volatility made the portfolios safer under the shock. The zero movement for conditional-volatility minimum variance is a result for this particular input and window, not a general guarantee of insensitivity.

### 3. Outlier contamination

The test replaces three of the 504 days with 8-standard-deviation shocks, repeated 40 times. All models returned valid results in all 40 runs (`n_ok = 40`), so there were no optimizer failures in these trials.

Common shocks generally perturb weights more than idiosyncratic shocks. Under common shocks, max-Sharpe again moves most on average (27.5% turnover equivalent; 50.7% at the 95th percentile) and has a 38.2% higher reference volatility. Risk parity, Black–Litterman, and conditional-volatility minimum variance have much smaller average weight changes (2.1%, 1.4%, and 0.9%). Under idiosyncratic shocks, average changes are smaller for most models, though max-Sharpe remains the most reactive on average (12.9%). The 95th-percentile figures also reveal occasional large responses—for example, conditional-volatility minimum variance reaches 29.7% under idiosyncratic shocks despite a 3.9% mean.

The main implication is that isolated extreme observations and broad common shocks can affect models differently; average sensitivity alone can hide a model’s tail response. These estimates are based on only 40 repetitions of one contamination design and should not be treated as universal rankings.

### 4. Direct covariance-spectrum perturbation

This experiment changes covariance eigenvalues and re-optimizes **GMV only**. As eigenvalue noise rises from 0.05 to 0.50, average weight movement grows from **0.79%** to **5.45%** turnover equivalent. Yet the variance ratio under the original covariance rises only from **1.0007** to **1.0309**—about **0.07% to 3.09% more variance**.

That gap is informative: weights can move while the portfolio’s objective value changes relatively little. Near a constrained minimum, the first-order change in the objective is zero along feasible directions, so the loss around the optimum can grow roughly quadratically with small weight deviations. The optimizer’s objective can therefore be locally flat even when the solution weights are not identical.

### 5. Historical scenarios

The weights are estimated using the 504 observations before each scenario, then applied through the episode without re-estimating. The reported cumulative returns are:

| Scenario | Equal weight | GMV | Max-Sharpe | Ledoit–Wolf | Risk parity | Robust MV |
|---|---:|---:|---:|---:|---:|---:|
| Q4 2018 selloff | −4.7% | +0.5% | −13.4% | 0.0% | −1.8% | −13.5% |
| COVID crash | −8.6% | +1.1% | +1.0% | +0.9% | −2.3% | −10.0% |
| 2022 rate shock | −18.4% | −15.3% | −14.7% | −15.0% | −16.1% | −14.6% |

The outcomes vary by episode: GMV and Ledoit–Wolf are comparatively resilient in Q4 2018 and COVID, while no model avoids substantial losses in the 2022 rate shock. Max-Sharpe and robust MV have especially poor Q4 2018 results, despite Max-Sharpe’s positive COVID result. This variability is a reason not to infer a generally superior strategy from these scenarios.

These calculations use constant target weights against daily returns (effectively maintaining/rebalancing to those weights in the calculation); they do not model transaction costs or adaptive trading within each episode.

### Scope and bottom line

Only **three of the five named stress windows** appear in the table: the dataset begins in 2012, so the earlier GFC and 2011 Euro-crisis windows lack the required history. Also, although mean-noise perturbation is listed in the notebook’s introduction, there is no corresponding experiment in the executed cells. The controlled experiments use one recent estimation window, and the historical comparison has only one outcome per episode; none establishes statistical significance or future performance.

Overall, the saved results point to **max-Sharpe as the most sensitive model in several tests**, **risk parity as relatively stable under these particular perturbations**, and **small GMV weight changes translating into modest risk penalties under the original covariance**. Treat these as diagnostic evidence about this sample and these perturbation designs, not as a definitive model ranking.